#Configuration

In [0]:
CATALOG = "cms_medicare_databricks_pipeline"
SOURCE_PROCEDURES = f"{CATALOG}.silver.procedures"
SOURCE_PROVIDERS  = f"{CATALOG}.silver.providers"
TARGET_COST       = f"{CATALOG}.gold.provider_cost_scorecard"

#Temp Views

In [0]:
spark.sql(f"CREATE OR REPLACE TEMP VIEW v_procedures AS SELECT * FROM {SOURCE_PROCEDURES}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW v_providers AS SELECT * FROM {SOURCE_PROVIDERS}")

print("Temp views created")
print(f"  v_procedures -> {SOURCE_PROCEDURES}")
print(f"  v_providers  -> {SOURCE_PROVIDERS}")

Temp views created
  v_procedures -> cms_medicare_databricks_pipeline.silver.procedures
  v_providers  -> cms_medicare_databricks_pipeline.silver.providers


#Gold Table: Provider Cost Scorecard

In [0]:
%sql
--Business Question: Which California providers have the highest cost variation for common procedures vs the state average?
CREATE OR REPLACE TABLE cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
-- Liquid clustering for query performance
CLUSTER BY (hcpcs_code, region, cost_outlier)
AS

WITH procedure_benchmarks AS (
    SELECT
        hcpcs_code,
        hcpcs_description,
        place_of_service,
        COUNT(DISTINCT provider_npi) AS total_providers_billing,
        ROUND(AVG(avg_medicare_standardized_amount), 2) AS state_avg_payment,
        ROUND(STDDEV(avg_medicare_standardized_amount), 2) AS state_stddev_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.25), 2) AS p25_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.50), 2) AS median_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.75), 2) AS p75_payment
    FROM v_procedures
    WHERE avg_medicare_standardized_amount IS NOT NULL
    GROUP BY all
),

provider_with_benchmarks AS (
    SELECT
        -- Provider Identity
        p.provider_npi,
        prov.provider_last_name,
        prov.provider_first_name,
        prov.credentials,
        prov.provider_type,
        prov.city,
        prov.zip_code,
        prov.ruca_code,
        -- Procedure identity
        p.hcpcs_code,
        p.hcpcs_description,
        p.place_of_service,
        -- Volume
        p.total_beneficiaries,
        p.total_services,
        -- Cost columns
        p.avg_submitted_charge,
        p.avg_medicare_allowed_amount,
        p.avg_medicare_payment_amount,
        p.avg_medicare_standardized_amount,
        -- State benchmarks for this procedure
        b.state_avg_payment,
        b.state_stddev_payment,
        b.median_payment,
        b.p25_payment,
        b.p75_payment,
        b.total_providers_billing,
        -- Dollar deviation from state average
        -- Positive = cost more than average, Negative = costs less than average
        ROUND(p.avg_medicare_standardized_amount - b.state_avg_payment, 2) AS deviation_from_state_avg,
        -- Percentage deviation from state average
        ROUND(((p.avg_medicare_standardized_amount - b.state_avg_payment) / NULLIF(b.state_avg_payment, 0)) * 100, 2) AS pct_deviation_from_state_avg,
        -- Flag cost outliers
        CASE
            WHEN p.avg_medicare_standardized_amount > b.p75_payment THEN 'High'
            WHEN p.avg_medicare_standardized_amount < b.p25_payment THEN 'Low'
            ELSE 'Normal'
        END AS cost_outlier,
        -- Orange County Flag
        CASE
            WHEN prov.zip_code LIKE '926%' 
                OR prov.zip_code LIKE '927%'
                OR prov.zip_code LIKE '928%' 
            THEN 'orange_county'
            ELSE 'other_region'
        END AS region,
        -- Billing ratio: what they submitted vs what Medicare paid
        ROUND(p.avg_submitted_charge / NULLIF(p.avg_medicare_payment_amount, 0),2) AS billing_ratio
    FROM v_procedures p
    INNER JOIN v_providers prov
        ON p.provider_npi = prov.provider_npi
    INNER JOIN procedure_benchmarks b
        ON p.hcpcs_code = b.hcpcs_code
        AND p.place_of_service = b.place_of_service
    WHERE p.avg_medicare_standardized_amount IS NOT NULL
)

-- Final output: all columns ordered by deviation descending
SELECT *
FROM provider_with_benchmarks
ORDER BY pct_deviation_from_state_avg DESC

num_affected_rows,num_inserted_rows


#Verify

In [0]:
%sql
-- Row counts by cost outlier category
SELECT
    cost_outlier,
    region,
    COUNT(*) AS row_count,
    ROUND(AVG(pct_deviation_from_state_avg), 2) AS avg_pct_deviation,
    ROUND(AVG(billing_ratio), 2)          AS avg_billing_ratio
FROM cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
GROUP BY cost_outlier, region
ORDER BY cost_outlier, region

cost_outlier,region,row_count,avg_pct_deviation,avg_billing_ratio
High,orange_county,18678,15.17,4.41
High,other_region,166343,14.52,5.38
Low,orange_county,15727,-17.66,6.47
Low,other_region,173798,-17.67,7.35
Normal,orange_county,43802,1.35,5.80
Normal,other_region,414191,1.42,7.05


### OC preview

In [0]:
%sql
-- Top 10 high cost Orange County providers
SELECT
    provider_last_name,
    provider_first_name,
    credentials,
    city,
    hcpcs_description,
    avg_medicare_standardized_amount,
    state_avg_payment,
    pct_deviation_from_state_avg,
    billing_ratio,
    cost_outlier
FROM cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
WHERE region = 'orange_county'
  AND cost_outlier = 'High'
ORDER BY pct_deviation_from_state_avg DESC
LIMIT 10

provider_last_name,provider_first_name,credentials,city,hcpcs_description,avg_medicare_standardized_amount,state_avg_payment,pct_deviation_from_state_avg,billing_ratio,cost_outlier
Mission Ambulatory Surgicenter Ltd.,null,null,Mission Viejo,Biopsy and aspiration of bone marrow sample for diagnosis,902.79,57.38,1473.35,4.32,High
Pegasus Surgery Center Llc,null,null,Newport Beach,Insertion of brain neurostimulator pulse device with connection to 2 or more electrode arrays,19847.74,1551.04,1179.64,2.52,High
"Main Street Specialty Surgery Center, Llc",null,null,Orange,Harvest of graft from small bone,3768.61,341.89,1002.29,2.15,High
Pavilion Surgery Center Llc,null,null,Orange,Insertion of pacemaker and upper and lower heart chamber electrode,5976.73,712.38,738.98,6.01,High
Specialty Surgical Center Of Irvine Lp,null,null,Irvine,Insertion of multicomponent inflatable penile implant,13010.24,1836.85,608.29,5.20,High
Orthopedic Surgery Center Of Oc Llc,null,null,Newport Beach,Replacement of thigh bone and hip joint with prosthesis,7280.09,1045.20,596.53,3.84,High
"The Surgery Center Of Newport Coast, Llc",null,null,Newport Beach,Replacement of thigh bone and hip joint with prosthesis,7239.78,1045.20,592.67,9.37,High
"California Specialty Surgery Center, Llc",null,null,Mission Viejo,Replacement of thigh bone and hip joint with prosthesis,7229.03,1045.20,591.64,3.74,High
You,Timothy,M.D.,Santa Ana,Unclassified drugs,2763.86,411.18,572.18,1.61,High
"Main Street Specialty Surgery Center, Llc",null,null,Orange,Harvest of graft from large bone,1329.91,201.92,558.63,7.65,High


#Optimize

In [0]:
%sql
-- Optimize for query performance and compaction
OPTIMIZE cms_medicare_databricks_pipeline.gold.provider_cost_scorecard

path,metrics
abfss://unity-catalog-storage@dbstorageu25jgq5vd3tok.dfs.core.windows.net/7405612093495677/__unitystorage/catalogs/14ce7132-4f74-43a5-af87-8b7608959a3b/tables/16ef1c13-43e5-4815-98e1-c2c132e1e05b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1783291185112, 1783291189478, 8, 0, null, List(0, 0), null, 28, 28, 0, 0, List(33178137, false, false, false, 0.0, List(0.0, 0.0, 0.0), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(155, 99, 1404, 0, 0, 0), 2, 1, 5, sizeAware, false, 0, null, false, 0, 0, 0, null, null, null), null)"
abfss://unity-catalog-storage@dbstorageu25jgq5vd3tok.dfs.core.windows.net/7405612093495677/__unitystorage/catalogs/14ce7132-4f74-43a5-af87-8b7608959a3b/tables/16ef1c13-43e5-4815-98e1-c2c132e1e05b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, true, 0, 0, 1783291189508, 1783291190736, 8, 0, null, List(0, 0), null, 28, 28, 0, 0, List(33178137, false, false, false, 0.0, List(0.0, 0.0, 0.0), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 513, 0, 0, 0), 15, 1, 1, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"
